
# Thí nghiệm 01 — Tổng hợp kết quả thực nghiệm Lorenz

## Mục tiêu

Notebook này tổng hợp các kết quả đã thu được từ quá trình huấn luyện và đánh giá ba biến thể mô hình dự báo chuỗi thời gian Lorenz:

1. PatchTST kết hợp toán tử Koopman.
2. Autoformer kết hợp toán tử Koopman.
3. Informer kết hợp toán tử Koopman.

Các kết quả được sử dụng để so sánh sai số dự báo, phân tích quá trình huấn luyện và đánh giá tính ổn định của toán tử Koopman.

## Nguồn dữ liệu kết quả

Toàn bộ số liệu trong notebook được trích xuất từ các output đã lưu trong:

`03_lorenz_training.ipynb`

Các checkpoint tương ứng nằm trong:

`results/`

Notebook không thực hiện huấn luyện, suy luận hoặc tính toán lại các metric. Những giá trị được trình bày là kết quả ghi nhận từ các lần thực nghiệm đã hoàn thành.

## Phạm vi

Notebook chỉ chứa bảng số liệu, giải thích và nhận xét kết quả.

Các hình minh họa tương ứng với bài báo gốc được quản lý riêng tại:

`05_lorenz_figures.ipynb`

## Lưu ý về phương pháp đánh giá

Các metric hiện có được tính trên chính các mẫu đã sử dụng trong quá trình huấn luyện.

Vì vậy, tất cả các giá trị MSE được trình bày là **in-sample MSE**, không phải test MSE trên tập dữ liệu độc lập.

Những kết quả này phản ánh mức độ khớp của mô hình với dữ liệu huấn luyện, chưa đủ để kết luận về khả năng tổng quát hóa.



## 1. Cấu hình thực nghiệm

### 1.1. Bộ dữ liệu Lorenz

Bộ dữ liệu được tạo từ hệ Lorenz có bổ sung nhiễu Gaussian trong quá trình mô phỏng.

Sau bước chuẩn bị dữ liệu bằng kỹ thuật cửa sổ trượt, các tensor có kích thước:

| Thành phần | Kích thước |
|---|---|
| Chuỗi Lorenz ban đầu | (2000, 3) |
| Dữ liệu đầu vào | (1845, 150, 3) |
| Nhãn dự báo | (1845, 15) |

Mỗi mẫu sử dụng 150 bước thời gian trong quá khứ để dự báo 5 bước tương lai của ba biến trạng thái.

### 1.2. Cấu hình huấn luyện

Ba mô hình sử dụng cùng cấu hình cơ bản:

| Tham số | Giá trị |
|---|---|
| Số epoch | 3000 |
| Số mẫu | 1845 |
| Batch size | 1845 (full-batch) |
| Bộ tối ưu | Adam |
| Learning rate | 0.001 |
| Hệ số chính quy hóa ổn định | 0.1 |
| Hạt giống ngẫu nhiên | 42 |

Quá trình huấn luyện sử dụng toàn bộ 1845 mẫu trong mỗi epoch.

### 1.3. Các kiến trúc được so sánh

| Mô hình | Bộ trích xuất đặc trưng |
|---|---|
| PatchTST + Koopman | `PatchTSTBackbone` |
| Autoformer + Koopman | `AutoformerBackbone` |
| Informer + Koopman | `InformerSparse` |

Cả ba mô hình đều kết hợp backbone với toán tử Koopman ổn định và bộ giải mã tuyến tính.

**Lưu ý:** Các backbone trong repository là những phiên bản triển khai đơn giản hóa. Không mặc nhiên coi chúng tương đương hoàn toàn với PatchTST, Autoformer và Informer gốc.



## 2. So sánh sai số dự báo

### 2.1. Chỉ số đánh giá

Thí nghiệm sử dụng sai số bình phương trung bình (*Mean Squared Error — MSE*):

$$
\mathrm{MSE}
=
\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2
$$

Trong đó:

- $\hat{y}_i$ là giá trị dự báo.
- $y_i$ là giá trị thực tế.
- $N$ là tổng số giá trị được so sánh.

MSE càng nhỏ cho thấy sai số bình phương trung bình trên tập dữ liệu được đánh giá càng thấp.

### 2.2. Kết quả thực nghiệm

**Bảng 1. So sánh sai số dự báo của ba mô hình trên dữ liệu Lorenz**

| Mô hình | In-sample MSE |
|---|---:|
| PatchTST + Koopman | **0.281637** |
| Autoformer + Koopman | 12.407236 |
| Informer + Koopman | 1.451354 |

*Nguồn: Output của cell so sánh ba mô hình trong `03_lorenz_training.ipynb`.*

### 2.3. Nhận xét

Từ các giá trị đã ghi nhận:

- **PatchTST + Koopman** có MSE nhỏ nhất trong ba mô hình, đạt 0.281637.
- **Informer + Koopman** đạt MSE bằng 1.451354, cao hơn PatchTST nhưng thấp hơn Autoformer.
- **Autoformer + Koopman** có MSE lớn nhất, đạt 12.407236 trong chế độ đánh giá.

Như vậy, trên bộ dữ liệu đã sử dụng để huấn luyện, biến thể PatchTST đạt sai số bình phương trung bình thấp nhất.

Tuy nhiên, do thí nghiệm chưa đánh giá trên tập kiểm tra độc lập, chưa thể khẳng định biến thể PatchTST có khả năng tổng quát hóa tốt nhất.

### 2.4. Nguồn checkpoint

Kết quả so sánh được tạo từ các checkpoint:

| Mô hình | Checkpoint |
|---|---|
| PatchTST | `patchtst_original_fullbatch.pt` |
| Autoformer | `autoformer_original_fullbatch.pt` |
| Informer | `informer_original_fullbatch.pt` |

Checkpoint Autoformer có hậu tố `(1)` là bản được huấn luyện trên Kaggle và sử dụng để tạo kết quả so sánh.

Sau khi xác minh và chuẩn hóa tên tệp, checkpoint này sẽ mang tên `autoformer_original_fullbatch.pt`, nhưng nội dung trọng số phải được bảo toàn.



## 3. Kết quả quá trình huấn luyện

### 3.1. Giá trị hàm mất mát

Hàm mất mát trong quá trình huấn luyện gồm sai số MSE kết hợp với thành phần chính quy hóa ổn định của toán tử Koopman.

Cả ba mô hình được huấn luyện trong 3000 epoch.

**Bảng 2. Giá trị hàm mất mát ở đầu và cuối quá trình huấn luyện**

| Mô hình | Loss ban đầu | Loss cuối cùng |
|---|---:|---:|
| PatchTST + Koopman | 278.294434 | 0.385890 |
| Autoformer + Koopman | 277.696991 | 3.928018 |
| Informer + Koopman | 282.345551 | 1.231271 |

*Nguồn: Output của các cell huấn luyện riêng cho từng mô hình.*

### 3.2. Nhận xét

Loss cuối cùng của cả ba mô hình thấp hơn đáng kể so với thời điểm bắt đầu huấn luyện.

Đối với PatchTST, loss cuối đạt 0.385890.

Đối với Autoformer, loss cuối đạt 3.928018.

Đối với Informer, loss cuối đạt 1.231271.

Điều này cho thấy các mô hình đã giảm được hàm mục tiêu trong quá trình tối ưu.

Tuy nhiên, lịch sử huấn luyện không giảm đơn điệu ở mọi thời điểm. Các output đã lưu cho thấy một số giai đoạn loss tăng trở lại, đặc biệt ở Autoformer và Informer.

### 3.3. Phân biệt training loss và MSE đánh giá

Training loss và in-sample MSE không nhất thiết bằng nhau do:

- Training loss có thể bao gồm thành phần chính quy hóa.
- Training loss được ghi nhận trong chế độ `model.train()`.
- MSE cuối được tính sau khi chuyển mô hình sang `model.eval()`.
- Các thành phần như Dropout có thể tạo ra sự khác biệt giữa hai chế độ.

Vì vậy, không được sử dụng loss cuối thay thế cho MSE khi lập bảng so sánh chất lượng dự báo.



## 4. Đánh giá tính ổn định của toán tử Koopman

### 4.1. Mục tiêu

Một thành phần quan trọng trong các mô hình là toán tử Koopman được xây dựng với ràng buộc ổn định.

Ma trận toán tử có dạng:

$$
K=U\operatorname{diag}(s)V^\top
$$

Trong đó $U$ và $V$ là các ma trận trực giao, còn các giá trị $s_i$ được khống chế bởi:

$$
0<s_i<0.99
$$

Cấu trúc này nhằm bảo đảm chuẩn phổ của toán tử nhỏ hơn 0.99.

### 4.2. Kết quả thực nghiệm

**Bảng 3. Giá trị kỳ dị lớn nhất của toán tử Koopman sau huấn luyện**

| Mô hình | Giá trị kỳ dị lớn nhất | Giới hạn |
|---|---:|---:|
| PatchTST + Koopman | 0.917746 | 0.99 |
| Autoformer + Koopman | 0.892004 | 0.99 |
| Informer + Koopman | 0.961025 | 0.99 |

*Nguồn: Output của cell tổng hợp độ ổn định Koopman trong `03_lorenz_training.ipynb`.*

### 4.3. Nhận xét

Cả ba giá trị kỳ dị lớn nhất đều nhỏ hơn ngưỡng 0.99.

Trong các giá trị được ghi nhận:

- Autoformer có giá trị kỳ dị lớn nhất thấp nhất.
- Informer có giá trị kỳ dị lớn nhất cao nhất.
- PatchTST nằm giữa hai mô hình còn lại.

Kết quả phù hợp với cách xây dựng toán tử Koopman có ràng buộc chuẩn phổ.

### 4.4. Giới hạn diễn giải

Giá trị kỳ dị lớn nhất và bán kính phổ là hai đại lượng khác nhau.

Cụ thể:

- Giá trị kỳ dị lớn nhất xác định chuẩn toán tử bậc hai.
- Bán kính phổ là độ lớn lớn nhất của các trị riêng.

Việc các giá trị kỳ dị đều nhỏ hơn 0.99 cho thấy toán tử có tính co theo chuẩn Euclid.

Tuy nhiên, điều kiện này không tự động bảo đảm toàn bộ mô hình dự báo đạt sai số thấp hoặc duy trì dự báo chính xác trong mọi chân trời thời gian.



## 5. Phân tích ảnh hưởng của Dropout đối với Autoformer

### 5.1. Hiện tượng quan sát

Trong quá trình thực nghiệm Autoformer, giá trị loss cuối cùng được ghi nhận xấp xỉ 3.928018.

Tuy nhiên, khi mô hình chuyển sang chế độ đánh giá bằng `model.eval()`, MSE thu được lớn hơn đáng kể.

Để tìm hiểu sự khác biệt này, notebook Training đã thực hiện các phép kiểm tra với chế độ huấn luyện, chế độ đánh giá và trạng thái Dropout.

### 5.2. Kết quả phân tích

**Bảng 4. So sánh MSE của Autoformer dưới các điều kiện hoạt động**

| Điều kiện | MSE |
|---|---:|
| Chế độ huấn luyện, Dropout bật | 3.976368 |
| Chế độ huấn luyện, Dropout tắt | 12.407235 |
| Chế độ đánh giá, Dropout tắt | 12.407235 |

*Nguồn: Output của hai cell phân tích train/eval và Dropout trong `03_lorenz_training.ipynb`.*

### 5.3. Nhận xét

Khi mô hình hoạt động ở chế độ huấn luyện với Dropout được bật, MSE ghi nhận xấp xỉ 3.976368.

Khi tắt Dropout, MSE tăng lên xấp xỉ 12.407235.

Trong phép kiểm tra đã thực hiện, MSE ở chế độ huấn luyện với Dropout tắt gần như bằng MSE ở chế độ đánh giá.

Điều này cho thấy sự thay đổi trạng thái Dropout có liên quan trực tiếp đến chênh lệch MSE quan sát được trong phép kiểm tra.

### 5.4. Giới hạn

Phân tích này chỉ xem xét hành vi của một mô hình Autoformer đã huấn luyện trên cùng bộ dữ liệu Lorenz.

Nó không đủ để kết luận Dropout luôn giúp tăng độ chính xác dự báo hoặc giải thích toàn bộ nguyên nhân khiến Autoformer có MSE cao hơn các mô hình còn lại.

Các kết quả vẫn mang tính in-sample.



## 6. Tổng kết kết quả thực nghiệm

### 6.1. Kết quả chính

Thí nghiệm đã hoàn thành quá trình huấn luyện ba biến thể mô hình dự báo Lorenz với cấu hình full-batch trong 3000 epoch.

Từ các kết quả được lưu lại:

1. PatchTST + Koopman đạt in-sample MSE thấp nhất, bằng 0.281637.
2. Informer + Koopman đạt in-sample MSE bằng 1.451354.
3. Autoformer + Koopman đạt in-sample MSE bằng 12.407236.
4. Giá trị kỳ dị lớn nhất của toán tử Koopman ở cả ba mô hình đều nhỏ hơn 0.99.
5. Autoformer xuất hiện sự khác biệt đáng kể giữa kết quả ở chế độ huấn luyện và chế độ đánh giá, có liên quan đến hoạt động của Dropout.

### 6.2. Giới hạn của thí nghiệm

Các hạn chế cần được làm rõ khi sử dụng kết quả trong báo cáo:

- Dữ liệu chưa được phân chia thành các tập huấn luyện và kiểm tra độc lập.
- Các giá trị MSE hiện có được tính trên tập dữ liệu đã dùng để huấn luyện.
- Các backbone là những phiên bản triển khai đơn giản hóa, không hoàn toàn tương ứng với kiến trúc chuẩn trong các bài báo gốc.
- Kết quả của Autoformer phụ thuộc vào checkpoint đã được huấn luyện trên Kaggle.
- Các metric MAE và RMSE từ cell dự báo PatchTST riêng chưa được đưa vào bảng tổng hợp do cần đối chiếu lại đối tượng mô hình tạo ra dự báo.

### 6.3. Hướng sử dụng kết quả

Các bảng trong notebook là nguồn số liệu cho phần kết quả và thảo luận của báo cáo.

Các hình minh họa được quản lý riêng trong `05_lorenz_figures.ipynb`.

Các nhận xét trong báo cáo cần dựa trên những kết quả đã ghi nhận, không tự bổ sung giá trị hoặc suy diễn về khả năng tổng quát hóa khi chưa có thực nghiệm kiểm chứng.

### 6.4. Tính toàn vẹn của kết quả

Notebook này chỉ tổng hợp số liệu từ output đã lưu, không thực thi lại mô hình.

Mọi thay đổi checkpoint hoặc đường dẫn sau này phải được kiểm tra để bảo đảm chúng không làm thay đổi kết quả thực nghiệm gốc.
